In [ ]:
!git clone -q https://github.com/shemshallah/nanlmv9.git
%cd nanlmv9
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!mkdir -p data/tok32k_wnmix
!curl -L --retry 3 -o data/tok32k_wnmix/meta.json https://github.com/shemshallah/nanlmv9/releases/download/natural-bytefirst/wnmix_meta.json
!for p in aa ab ac ad; do curl -L --retry 3 -o cmnat.part-$p https://github.com/shemshallah/nanlmv9/releases/download/natural-bytefirst/cmnat.part-$p; done
!cat cmnat.part-* > cm_train_natural_512_ctx17.bin
!curl -L --retry 3 -o win_train_natural_512.bin https://github.com/shemshallah/nanlmv9/releases/download/natural-bytefirst/win_train_natural_512.bin
!curl -L --retry 3 -o word_prior_wnmix.npy https://github.com/shemshallah/nanlmv9/releases/download/natural-bytefirst/word_prior_wnmix.npy
!for p in aa ab ac ad ae af; do curl -L --retry 3 -o cm17.part-$p https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-ctx17/cm17.part-$p; done
!cat cm17.part-* > cm_valid_full_ctx17.bin
!curl -L --retry 3 -o win_valid_full.bin https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-ctx17/win_valid_full.bin
!ls -lh cm_train_natural_512_ctx17.bin win_train_natural_512.bin cm_valid_full_ctx17.bin word_prior_wnmix.npy

In [ ]:
!python3 experiments/mamba_moe/train_word_char.py \
  --data-dir data/tok32k_wnmix --cm-dump cm_train_natural_512_ctx17.bin --window win_train_natural_512.bin \
  --word-prior word_prior_wnmix.npy \
  --eval-window win_valid_full.bin --eval-cm-dump cm_valid_full_ctx17.bin --eval-chunks 20 \
  --out runs/d256_natural --steps 100000 --tokens 64 \
  --d-model 256 --char-model 64 --experts 64 --lr 2e-4 \
  --scan chunked --scan-chunk 4 --device cuda --amp --max-seconds 600 \
  --save-interval 500 --result-json runs/d256_natural/result.json

In [ ]:
import json; print(open('runs/d256_natural/result.json').read())